# 암환자 유전체 데이터 기반 암종 분류 — 메인 파이프라인

공식 베이스라인(`00_official_baseline_xgb.ipynb`)의 5단계 구조를 그대로 따른다.

| 단계 | 공식 베이스라인 | 이 노트북 |
|---|---|---|
| Load Data | train + test 동시 로드 | **train만** 로드 |
| Preprocessing | OrdinalEncoder(변이 문자열→정수) | `src/features.py` 피처 (official / v1 선택) |
| Model Train | XGB 100트리, 검증 없음 | XGB + **Stratified 5-Fold** OOF Macro F1 / Acc |
| Inference | test 인코딩 → predict | 전체 train 재학습 → **여기서만 test 로드** → predict |
| Submission | baseline_submission.csv | `submissions/{날짜}_{피처}_xgb.csv` |

> ⚠️ 규칙: test.csv는 Inference 셀 이전에 절대 읽지 않는다.

# Import library

In [ ]:
import sys, json
sys.path.insert(0, "../src")
import pandas as pd
from main import load_train, cross_validate, fit_full_and_submit, XGB_PARAMS, ROOT
from datetime import date

FEATURES = "v1"          # "official" 로 바꾸면 공식 베이스라인 인코딩 재현
TAG = f"{date.today().isoformat()}_{FEATURES}_xgb"
print(XGB_PARAMS)

# Load Data

In [ ]:
train = load_train()
print(train.shape)
train.iloc[:3, :8]

# Data Preprocessing

피처 생성 로직은 `src/features.py`에 있다. fit이 필요한 통계는 fold 안의 train 부분에서만 계산된다.

In [ ]:
from main import FeatureMaker
fm = FeatureMaker(FEATURES).fit(train)
X_preview = fm.transform(train.head(5))
print(X_preview.shape)
X_preview.iloc[:, -12:]

# Model Define and Train (Stratified 5-Fold)

In [ ]:
res = cross_validate(train, FEATURES, XGB_PARAMS, out_dir=ROOT / "experiments" / TAG)
pd.DataFrame(res["folds"])

In [ ]:
# 클래스별 F1 (낮은 순)
pd.Series(res["per_class_f1"]).sort_values().head(10)

# Inference

전체 train으로 재학습한 뒤 **이 셀에서 처음으로** test.csv를 읽는다.

In [ ]:
out = fit_full_and_submit(train, FEATURES, XGB_PARAMS, TAG)

# Submission

In [ ]:
sub = pd.read_csv(out)
print(out, sub.shape)
sub.head()